# Lab 2 — 도구 사용 (Tool Use)

> **이론 복습 — Session 2 슬라이드**
> - 도구(Tool) = LLM 이 호출할 수 있는, 우리가 만든 함수
> - Function calling 4단계: ① 스키마 전달 → ② 모델이 호출 결정 → ③ 우리가 실행 → ④ 결과 반환
> - LLM 은 *결정* 만 한다. 실행은 *우리 코드* 가 한다.

## lab0 → lab1 → lab2 의 흐름

lab0 챗봇은 BUILD_SPEC §4-5 첫 줄을 따라 **모든 데이터를 시스템 프롬프트에 통째로 박았습니다**
(`src/lib/chat.js` ▶ `buildSystemPrompt()`). lab1 에서 그 방식을 Python 으로 재현해 봤죠.

그런데 이 방식은 한계가 있습니다:
- 데이터가 커지면 프롬프트가 폭발한다
- 새 계산이 필요하면 매번 미리 계산해서 박아야 한다
- 모델이 *언제* 어떤 계산을 했는지 추적하기 어렵다

**lab2 의 답**: 데이터를 전부 박지 말고, LLM 에게 **"필요할 때 부를 함수 목록"** 만 알려 준다.
그게 function calling 이다.

## 학습 목표
1. lab0 의 통근 데이터로, 도구가 "그냥 Python 함수" 임을 확인한다
2. 도구 **JSON 스키마** 를 직접 작성한다
3. Function calling **한 바퀴** (①→④) 를 손으로 돌려본다
4. 🔧 새 도구를 스키마부터 추가한다


## 0. 준비

- `labs/` 폴더에서 이 노트북을 실행하세요.
- `.env` 에 `GEMINI_API_KEY` 가 있으면 실제 모델, 없으면 `MockLLM` 으로 동작합니다.


In [ ]:
import json
from pathlib import Path
from common.llm import LLMClient

llm = LLMClient()

# lab0 데이터 — lab1 과 똑같이 읽어 옵니다.
DATA = Path("lab0_vibe_coding/data")
cities = json.loads((DATA / "cities.json").read_text(encoding="utf-8"))
flows  = json.loads((DATA / "flows.json").read_text(encoding="utf-8"))

# 자주 쓸 인덱스 — id → 도시 dict, 한글이름 → id
CITY_BY_ID   = {c["id"]: c for c in cities}
CITY_BY_NAME = {c["name_ko"]: c["id"] for c in cities}

print(f"도시 {len(cities)}개, 통근 흐름 {len(flows)}개")

## 1. 도구는 "그냥 함수" 다

거창한 게 아닙니다. 도구는 **입력을 받아 결과를 돌려주는 평범한 Python 함수**입니다.
lab0 의 `src/lib/stats.js` 에 있던 `cityTotals`, `busiestFlow` 와 형제 같은 함수들입니다.

핵심 차이는 단 하나 — 이번엔 **LLM 이 이 함수들을 부른다**는 것뿐입니다.


In [ ]:
def find_city(query: str) -> list[dict]:
    """이름/한글이름/id 어디든 query 가 들어 있는 도시를 찾는다.
    lab0 의 ChatPanel.jsx ▶ findMentionedCities() 가 하던 일과 같다 —
    이번엔 LLM 이 호출할 수 있게 *함수* 로 노출한다."""
    q = (query or "").strip().lower()
    hits = [
        {"id": c["id"], "name_ko": c["name_ko"], "name": c["name"],
         "lat": c["lat"], "lon": c["lon"]}
        for c in cities
        if q and (q in c["name_ko"].lower() or q in c["name"].lower() or q == c["id"].lower())
    ]
    return hits


def _resolve(name_or_id: str) -> str | None:
    """한글이름이든 id 든 받아 id 로 정규화."""
    if not name_or_id:
        return None
    if name_or_id in CITY_BY_ID:
        return name_or_id
    return CITY_BY_NAME.get(name_or_id)


def get_flow(origin: str, dest: str) -> dict:
    """단방향 통근량 — origin → dest. lab0 의 첫 예시(서울→인천 520)와 같은 종류의 질의."""
    oid, did = _resolve(origin), _resolve(dest)
    if not oid or not did:
        return {"error": f"city not found: origin={origin}, dest={dest}"}
    for f in flows:
        if f["origin"] == oid and f["dest"] == did:
            return {"origin": oid, "dest": did, "count": f["count"]}
    return {"origin": oid, "dest": did, "count": 0}


def get_city_totals(city: str) -> dict:
    """한 도시의 inflow / outflow / total. lab0 의 stats.js ▶ cityTotals() 중 한 줄과 같다."""
    cid = _resolve(city)
    if not cid:
        return {"error": f"city not found: {city}"}
    inflow  = sum(f["count"] for f in flows if f["dest"]   == cid)
    outflow = sum(f["count"] for f in flows if f["origin"] == cid)
    return {"city": cid, "name_ko": CITY_BY_ID[cid]["name_ko"],
            "inflow": inflow, "outflow": outflow, "total": inflow + outflow}


def get_top_flows(n: int = 5) -> list[dict]:
    """통근량 상위 N 개 흐름. lab0 의 buildStatsSummary() 의 busiest 자리에 대응."""
    return [
        {"origin": f["origin"], "dest": f["dest"], "count": f["count"],
         "label": f"{CITY_BY_ID[f['origin']]['name_ko']}→{CITY_BY_ID[f['dest']]['name_ko']}"}
        for f in sorted(flows, key=lambda x: -x["count"])[:n]
    ]


# 도구는 그냥 함수다 — 직접 불러서 확인.
print("find_city('서울'):",      find_city("서울"))
print("get_flow('서울','인천'):", get_flow("서울", "인천"))
print("get_city_totals('서울'):", get_city_totals("서울"))
print("get_top_flows(3):")
for f in get_top_flows(3):
    print("  ", f)

## 2. 도구 스키마 정의 (① 단계)

LLM 은 함수의 **코드** 를 보지 못합니다. **스키마**(이름·설명·파라미터) 만 봅니다.
`description` 은 *모델이 읽는 안내문* 이라는 점에 주목하세요 — 거기에 적은 글이 곧 모델의 판단 근거입니다.


In [ ]:
SCHEMA_FIND_CITY = {
    "name": "find_city",
    "description": "Find cities whose Korean name, English name, or id contains the query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Part of a city name, e.g. '서울' or 'SEO'."},
        },
        "required": ["query"],
    },
}

SCHEMA_GET_FLOW = {
    "name": "get_flow",
    "description": "Directed commute volume from origin to dest. "
                   "origin/dest may be Korean names ('서울') or ids ('SEO'). "
                   "Note: SEO→INC and INC→SEO are different items.",
    "parameters": {
        "type": "object",
        "properties": {
            "origin": {"type": "string"},
            "dest":   {"type": "string"},
        },
        "required": ["origin", "dest"],
    },
}

SCHEMA_CITY_TOTALS = {
    "name": "get_city_totals",
    "description": "Total commute inflow / outflow / sum for one city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name or id."},
        },
        "required": ["city"],
    },
}

SCHEMA_TOP_FLOWS = {
    "name": "get_top_flows",
    "description": "Return the N largest commute flows.",
    "parameters": {
        "type": "object",
        "properties": {
            "n": {"type": "integer", "description": "How many flows to return (default 5)."},
        },
        "required": [],
    },
}

my_tools = [SCHEMA_FIND_CITY, SCHEMA_GET_FLOW, SCHEMA_CITY_TOTALS, SCHEMA_TOP_FLOWS]
TOOLBOX = {
    "find_city":       find_city,
    "get_flow":        get_flow,
    "get_city_totals": get_city_totals,
    "get_top_flows":   get_top_flows,
}
print(f"{len(my_tools)} tool schemas ready.")

## 3. Function calling 한 바퀴

lab0 와 똑같은 질문 — "서울에서 인천으로 가는 통근량은?" — 인데, 이번엔 **데이터를 안 박습니다**.
대신 도구 스키마만 주고, 모델이 어떤 도구를 어떻게 부를지 보겠습니다.


In [ ]:
question = "서울에서 인천으로 가는 통근량은?"
messages = [{"role": "user", "content": question}]

reply = llm.generate_with_tools(messages, my_tools)   # ① + ②
print("도구를 부르려 하나요?", reply.wants_tool)
for call in reply.tool_calls:
    print("  tool_call:", call.name, call.args)

모델이 일반 텍스트가 아니라 **구조화된 호출**(`tool_call`)로 응답했습니다.
이제 ③ 우리가 실행하고, ④ 결과를 모델에 돌려줍니다.


In [ ]:
# ③ WE execute the tool the model asked for
call = reply.tool_calls[0]
result = TOOLBOX[call.name](**(call.args or {}))
print("도구 실행 결과:", result)

# ④ send the result back, so the model can write the final answer
messages.append({"role": "assistant", "content": reply.text,
                 "tool_calls": reply.tool_calls})
messages.append({"role": "tool", "name": call.name,
                 "content": json.dumps(result, ensure_ascii=False)})

final = llm.generate_with_tools(messages, my_tools)
print()
print("최종 답변:", final.text)

## 3-bonus. 시스템 프롬프트로 도구 사용을 *강제* 하기

같은 질문이어도 **시스템 프롬프트** 한 줄에 따라 모델의 판단이 바뀝니다.
`generate_with_tools(..., system="...")` 로 넣어 보면 도구를 부르거나, 안 부르거나,
"모르면 모른다고 답하라" 같은 행동까지 제어할 수 있습니다.

> 시스템 프롬프트가 *권유* 라면, Gemini 의 `FunctionCallingConfig.mode="ANY"` 는 *강제* 입니다
> (텍스트 답변 자체를 금지). 이번 워크샵에선 권유 수준으로 충분합니다.


In [ ]:
# 모델이 그냥 잡담으로 답할 수도 있는 모호한 질문
q = "서울은 어떤 도시야?"

# [A] 시스템 프롬프트 없음 — 모델이 알아서 판단 (보통 학습 지식으로 텍스트 답변)
r1 = llm.generate_with_tools([{"role": "user", "content": q}], my_tools)
print("[A] no system  → wants_tool:", r1.wants_tool)
print("   text:", (r1.text or "")[:80])
for c in r1.tool_calls:
    print("   tool_call:", c.name, c.args)

print()

# [B] 시스템 프롬프트로 도구 사용 강제
strict_system = (
    "너는 수도권 통근 데이터 분석가다. "
    "도시·통근량과 조금이라도 관련 있는 질문은 반드시 제공된 도구를 호출해 답하라. "
    "학습 지식이나 추측으로 답하지 말 것. 데이터에 없으면 '데이터에 없음' 이라고 답하라."
)
r2 = llm.generate_with_tools([{"role": "user", "content": q}], my_tools,
                              system=strict_system)
print("[B] strict sys → wants_tool:", r2.wants_tool)
print("   text:", (r2.text or "")[:80])
for c in r2.tool_calls:
    print("   tool_call:", c.name, c.args)

**lab0 와 비교해 보세요.**

- lab0: cities 18개 + flows 48개 + stats 가 **시스템 프롬프트에 통째로** 들어 있었다.
- lab2: 시스템 프롬프트는 비어 있다. 대신 **도구 스키마 4개** 만 보여 줬다.
  필요한 한 줄(서울→인천)만 우리가 꺼내 줬다.

도시가 1,800 개가 돼도 이 방식은 그대로 동작합니다.


## 4. 한 함수로 묶기

①~④ 를 `one_round()` 함수 하나로 묶어 두면 재사용하기 편합니다.
(lab3 에서는 이걸 *여러 번 반복* 하는 루프로 발전시킵니다.)


In [ ]:
def one_round(question, schemas):
    """Run one tool-use round-trip: ask -> (maybe call a tool) -> answer."""
    messages = [{"role": "user", "content": question}]
    reply = llm.generate_with_tools(messages, schemas)

    if not reply.wants_tool:
        return reply.text                       # model answered directly

    call = reply.tool_calls[0]
    result = TOOLBOX[call.name](**(call.args or {}))
    messages.append({"role": "assistant", "content": reply.text,
                     "tool_calls": reply.tool_calls})
    messages.append({"role": "tool", "name": call.name,
                     "content": json.dumps(result, ensure_ascii=False)})
    return llm.generate_with_tools(messages, schemas).text


# lab0 의 BUILD_SPEC §4-4 예시 대화 중 단일-도구로 풀리는 것들
print(one_round("통근량이 가장 많은 구간은?", my_tools))
print()
print(one_round("용인이라는 도시가 데이터에 있어?", my_tools))

## 🔧 TODO — 새 도구를 스키마부터 추가하기

BUILD_SPEC §4-4 의 마지막 예시는 **"수원과 용인 사이 왕복 통근량은?" (정답 510)** 입니다.
현재 `get_flow` 로 두 번 부르면 풀 수 있지만, **한 번에** 풀어 주는 도구가 있으면 편하겠죠.

`get_round_trip(city_a, city_b)` 라는 **새 도구** 를 만들어 보세요.
반환은 `{a_to_b, b_to_a, total}` 모양이면 좋습니다.


In [ ]:
# 🔧 TODO 1: 함수 본체를 완성하세요.
def get_round_trip(city_a: str, city_b: str) -> dict:
    # 힌트: get_flow(a, b) 와 get_flow(b, a) 의 count 를 더한다.
    pass


# 🔧 TODO 2: 위 함수의 스키마를 완성하세요.
SCHEMA_ROUND_TRIP = {
    "name": "get_round_trip",
    "description": "",          # TODO: 도구가 하는 일을 영어로 적으세요
    "parameters": {
        "type": "object",
        "properties": {
            # TODO: city_a 와 city_b 를 정의하세요 (둘 다 "string")
        },
        "required": [],         # TODO: 필수 인자 이름을 적으세요
    },
}

# 🔧 TODO 3: TOOLBOX 에 등록하세요.
# TOOLBOX["get_round_trip"] = ...

# --- test (정답: 510) ---
q = "수원과 용인 사이 왕복 통근량은?"
print(one_round(q, my_tools + [SCHEMA_ROUND_TRIP]))

## 정리

- 도구 = 평범한 함수 + 그것을 설명하는 **스키마**
- Function calling 한 바퀴: 스키마 전달 → 호출 결정 → 실행 → 결과 반환
- LLM 은 *무엇을 부를지* 만 정하고, 실행은 우리 코드가 한다
- **lab0 와의 진짜 차이**: 데이터를 *전부* 박지 않고, 모델이 *필요한 한 줄* 만 꺼내 쓰게 한다

**다음 — Session 3 (에이전트 루프)**
> 한 바퀴로는 풀리지 않는 질문이 있다: *"통근량이 가장 많은 구간을 찾고, 그 두 도시의 거리도 알려줘"* — 도구를 **두 번 이상** 불러야 한다. lab3 에서는 이 한 바퀴를 *여러 번 반복* 하는 **에이전트 루프** 를 만듭니다.
